# « Développez une preuve de concept »

Projet n&#8239;$^\text{o}$ 7 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 18/05/2026

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

# Notebook d’entraînement du modèle SoTA `YOLO26` à 3 classes

Le dataset : http://vision.stanford.edu/aditya86/ImageNetDogs/

## Imports et constantes

In [ ]:
from functions_model_building import (
    plot_confusion_matrix,
    print_classification_report
)
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import datetime as dt
import itertools as it
import logging
import numpy as np
import os
import pandas as pd
import pytz
import re
import time
import torch

warnings = __import__('warnings')
warnings.filterwarnings("ignore")

LOGGER_FORMAT = '%(asctime)s [%(levelname)s] %(message)s'
logging.Formatter.converter = lambda *_: dt.datetime.now(pytz.timezone('Europe/Paris')).timetuple()
logging.basicConfig(level=logging.INFO, format=LOGGER_FORMAT, force=True)
logr = logging.getLogger(__name__)
logr.setLevel(logging.DEBUG)

BREED_ID_SPLITTER_REGEX_PTRN = re.compile(r'^n\d+-')
DEFAULT_BATCH_SIZE = 8
DEFAULT_TOP_K_CATEG_ACCUR = 5
DEFAULT_TRGT_IMG_SIZE = (224, 224)
DO_NOT_AUTOSAVE = -1
LEARNING_RATE_LOW = 1e-5
LEARNING_RATE_REGULAR = 1e-2
N_EPOCHS = 20
PRETR_MDL_LBL = 'yolo26n-cls'

## Split train - val - test

In [ ]:
input_img_paths = tuple(
    [Path('images/n02090379-redbone'), Path('images/n02085936-Maltese_dog'), Path('images/n02088094-Afghan_hound')]
    # Path('images/').glob('*/')
)
logr.info(f"Nombre de classes retenues pour l'entraînement : {len(input_img_paths)}")
input_img_paths = tuple(path.glob('*.jpg') for path in input_img_paths)
input_img_paths = tuple(it.chain.from_iterable(input_img_paths))
logr.info(f"Résultat : un tuple de {len(input_img_paths)} images pour les 3 classes de races.")
breed_labels = tuple(str(path).split('/')[1] for path in input_img_paths)
breed_labels_encoded = LabelEncoder().fit_transform(breed_labels)
breed_labels_dict = dict(sorted(zip(breed_labels_encoded, breed_labels)))
n = 1000
logr.info(f"Un échantillon de la distribution des classes (seuls les {n} premiers chemins sont comptés) :")
display(dict(Counter(breed_labels[:n])))
train_frac = 0.8
input_paths_train, input_paths_val_test, labels_train, labels_val_test = train_test_split(
    input_img_paths, breed_labels, train_size=train_frac,
    random_state=42, shuffle=True, stratify=breed_labels,
)
val_test_frac = 0.5
input_paths_val, input_paths_test, labels_val, labels_test = train_test_split(
    input_paths_val_test, labels_val_test, train_size=val_test_frac,
    random_state=42, shuffle=True, stratify=labels_val_test
)
del input_paths_val_test, labels_val_test, n
logr.info(f"Taile de l'échantillon de train ({train_frac}): {len(input_paths_train)}")
logr.info(f"Taile de l'échantillon de val ({round(val_test_frac * (1 - train_frac), 1)}): {len(input_paths_val)}")
logr.info(f"Taile de l'échantillon de test ({round(val_test_frac * (1 - train_frac), 1)}): {len(input_paths_test)}")

## Entraînement

### Déclarer les paramètres

In [ ]:
N_CLASSES = len(set(breed_labels))
CLASS_NAMES_SORTED = sorted(set(breed_labels))
FT = 10
TOP_K_CATEG_ACCUR = min(DEFAULT_TOP_K_CATEG_ACCUR, N_CLASSES - 1)

### Training: Stage 1: freeze backbone, train head and neck

In [ ]:
experiment_name = (
    f'CNN__from={PRETR_MDL_LBL}'
    f'__n_cls={N_CLASSES}'
    f'__n_eps={N_EPOCHS}'
    f'__LR={LEARNING_RATE_REGULAR}'
    f'__FT={FT}'
    f'__stage1'
)
PATH_TO_MDL_STG1 = f'models/{experiment_name}.pt'
if os.path.exists(PATH_TO_MDL_STG1):
    logr.info(f"Loading saved model {PATH_TO_MDL_STG1!r}")
    model = YOLO(PATH_TO_MDL_STG1)
else:
    logr.info(f"Training {experiment_name}:",
              "Stage 1: freeze backbone, train head and neck")
    PATH_TO_ORIG_MDL = f'models/{PRETR_MDL_LBL}.pt'
    model = YOLO(PATH_TO_ORIG_MDL)
    results = model.train(
        data={
            'train': input_paths_train.tolist(),
            'val': input_paths_val.tolist(),
            'test': input_paths_test.tolist(),
            'nc': N_CLASSES,
            'names': CLASS_NAMES_SORTED
        },
        epochs=N_EPOCHS,
        imgsz=DEFAULT_TRGT_IMG_SIZE[0],
        batch=DEFAULT_BATCH_SIZE,
        freeze=FT,
        lr0=LEARNING_RATE_REGULAR,
        name=experiment_name,
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=True,
        project=f"log/{experiment_name}",
        save_period=DO_NOT_AUTOSAVE,
        exist_ok=True,
    )
    logr.info("Training Stage 1 complete.",
              f"Saving model to {PATH_TO_MDL_STG1!r}")
    model.save(PATH_TO_MDL_STG1)

### Training: Stage 2: unfreeze all, fine-tune with lower learning rate

In [ ]:
torch.cuda.empty_cache()

In [ ]:
experiment_name = (
    f'CNN__from={PRETR_MDL_LBL}'
    f'__n_cls={N_CLASSES}'
    f'__n_eps={N_EPOCHS}'
    f'__LR={LEARNING_RATE_LOW}'
    f'__FT={FT}'
    f'__stage2'
)
PATH_TO_MDL_STG2 = f'models/{experiment_name}.pt'
if os.path.exists(PATH_TO_MDL_STG2):
    logr.info(f"Loading saved model {PATH_TO_MDL_STG2!r}")
    model = YOLO(PATH_TO_MDL_STG2)
else:
    logr.info(f"Training {experiment_name}:",
              "Stage 2: unfreeze whole backbone, fine-tune with lower LR")
    model = YOLO(PATH_TO_MDL_STG1)
    results = model.train(
        data={
            'train': input_paths_train.tolist(),
            'val': input_paths_val.tolist(),
            'test': input_paths_test.tolist(),
            'nc': N_CLASSES,
            'names': CLASS_NAMES_SORTED
        },
        epochs=N_EPOCHS,
        imgsz=DEFAULT_TRGT_IMG_SIZE[0],
        batch=DEFAULT_BATCH_SIZE,
        lr0=LEARNING_RATE_LOW,
        name=experiment_name,
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=True,
        project=f"log/{experiment_name}",
        save_period=DO_NOT_AUTOSAVE,
        exist_ok=True,
    )
    logr.info("Training Stage 2 complete.",
              f"Saving model to {PATH_TO_MDL_STG2!r}")
    model.save(PATH_TO_MDL_STG2)

## Évaluation sur les données de test

In [ ]:
torch.cuda.empty_cache()

### Metrics

In [ ]:
logr.info("Evaluating on test set...")
metrics = model.val(
    data={
        'val': input_paths_val.tolist(),
        'test': input_paths_test.tolist(),
        'nc': N_CLASSES,
        'names': CLASS_NAMES_SORTED
    },
    split='test',
    batch=DEFAULT_BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
)
logr.info(f"Metrics :\n\n{metrics}")
display(metrics)

### Classification Report & Confusion Matrix

In [ ]:
y_true = []
y_pred_probs = []
y_pred_topk = []
for path, label in zip(input_paths_test, labels_test):
    results = model(path, verbose=False)
    probs = results[0].probs.cpu().numpy()
    y_pred_probs.append(probs)
    true_idx = CLASS_NAMES_SORTED.index(label)
    y_true.append(true_idx)
    top_k_idx = np.argpartition(probs, -TOP_K_CATEG_ACCUR)[-TOP_K_CATEG_ACCUR:]
    y_pred_topk.append(true_idx in top_k_idx)
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred_probs)
print("\nClassification Report:\n")
print_classification_report(y_true_arr, y_pred_arr, class_labels=CLASS_NAMES_SORTED)
print("\nConfusion Matrix:\n")
plot_confusion_matrix(y_pred_arr, y_true_arr, CLASS_NAMES_SORTED, title=experiment_name)
top_k_accuracy = sum(y_pred_topk) / len(y_pred_topk)
print(f"\nTop-k Accuracy: {top_k_accuracy}\n")

## Prédiction en mode *blind test*

Effectuer l'inférence :

In [ ]:
DEFAULT_BLIND_TEST_SAMPLE_SIZE = 100
blind_test_sample = np.random.choice(
    input_img_paths,
    size=DEFAULT_BLIND_TEST_SAMPLE_SIZE,
    replace=False
)
start_time = time.perf_counter()
predicted_class_idx = []
for path in blind_test_sample:
    results = model(path, verbose=False)
    pred_idx = results[0].probs.top5[0]
    predicted_class_idx.append(pred_idx)
end_time = time.perf_counter()
print(f"\nInference time : {end_time - start_time:.2f}s")

Afficher les résultats sous une forme conviviale :

In [ ]:
df = pd.DataFrame({
    'path': blind_test_sample,
    'predicted_breed': [BREED_ID_SPLITTER_REGEX_PTRN.split(CLASS_NAMES_SORTED[pred_idx])[1]
                        for pred_idx in predicted_class_idx]
}, dtype=str)
df['is_correct'] = df.apply(
    lambda row: '✅' if row['predicted_breed'] in row['path'] else '❗️',
    axis=1
)
default_max_rows = pd.options.display.max_rows
pd.options.display.max_rows = DEFAULT_BLIND_TEST_SAMPLE_SIZE
display(df)
pd.options.display.max_rows = default_max_rows